In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pdb
from IPython.display import display
import os

In [34]:
import ee
import geemap
service_account = ' boba-account@boba-430314.iam.gserviceaccount.com '
credentials = ee.ServiceAccountCredentials(service_account,'C:/Users/arj26323/OneDrive - University of Georgia/Documents/GEE AUTH/boba-430314-a19be859bfca.json')

ee.Initialize(credentials)


# ee.Authenticate()
# ee.Initialize()



print(geemap.__version__)

0.34.1


In [35]:
# 31.32615  -81.32228333
df = pd.read_csv('C:/Users/arj26323/Documents/Data/GA_conservation/fixed_actives_installations.csv')
path = 'C:/Users/arj26323/Documents/Data/GA_conservation/fixed_actives_installations.csv'
# df

fc_all = geemap.csv_to_ee(
    path, 
    latitude="Latitude", 
    longitude="Longitude"
)

alta = ee.Feature(fc_all.toList(1, 4).get(0))

In [40]:
alta_buffer = alta.buffer(2000)

naip = ee.ImageCollection('USDA/NAIP/DOQQ').filterBounds(alta.geometry()).filter(ee.Filter.date('2017-01-01', '2018-12-31'))

naip_image = ee.Image(naip.first())
naip_mosaic = naip.mosaic().clip(alta.buffer(2000).geometry())

Map(center=[31.32615, -81.32228333], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Se…

In [41]:
print('Object type:', type(naip_mosaic))
print('Number of NAIP images found:', naip.size().getInfo())

Object type: <class 'ee.image.Image'>
Number of NAIP images found: 1


In [73]:
# Check availability for early 2000s
naip_early = ee.ImageCollection('USDA/NAIP/DOQQ') \
    .filterBounds(alta.geometry()) \
    .filter(ee.Filter.date('1999-01-01', '2005-12-31'))

# Print the available years/dates
dates = naip_early.aggregate_array('system:time_start')
print('Available NAIP dates (2000-2005):', dates.map(lambda d: ee.Date(d).format('YYYY-MM-dd')).getInfo())


naip_early_mosaic = naip_early.mosaic().clip(alta.buffer(2000).geometry())

Available NAIP dates (2000-2005): ['2005-10-16']


In [74]:
l03_vis = l03.select(['SR_B4','SR_B3','SR_B2']) \
            .multiply(0.0000275) \
            .add(-0.2)


Map = geemap.Map(center=[31.32615, -81.32228333], zoom=14)
Map.addLayer(alta_buffer, {}, 'buffer')

# l03 = ee.Image('LANDSAT/LT05/C02/T1_L2/LT05_016038_20030127').clip(alta.buffer(2000).geometry())

# l03 = ee.Image('LANDSAT/LC08/C02/T1_L2/LC08_016038_20170101').clip(alta.buffer(2000).geometry())


'''
LANDSAT/LT05/C02/T1_L2/LT05_016038_20010105
LANDSAT/LT05/C02/T1_L2/LT05_016038_20040302




LANDSAT/LT05/C02/T1_L2/LT05_016038_20100522
LANDSAT/LC08/C02/T1_L2/LC08_016038_20170322





LANDSAT/LT05/C02/T1_L2/LT05_016038_20030519
LANDSAT/LT05/C02/T1_L2/LT05_016038_20010310


LANDSAT/LT05/C02/T1_L2/LT05_016038_20100319
LANDSAT/LC08/C02/T1_L2/LC08_016038_20170101 L8
'''

Map.addLayer(naip_early_mosaic, {'bands': ['R', 'G', 'B']}, 'NAIP 2005')
Map.addLayer(naip_mosaic, {'bands': ['R', 'G', 'B']}, 'NAIP 2017')


# Map.addLayer(l03_vis,
#     {
#         'min': 0.0,
#         'max': 0.3
#     },
#     'Landsat 2003')

Map.addLayer(alta, {}, 'Estuaries Areas of the Altamaha River')

Map

Map(center=[31.32615, -81.32228333], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Se…